<a href="https://colab.research.google.com/github/prasa129/Fun/blob/main/YTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estimating YTM

10-25-2025

Standard bond pricing formulas present price as a function of a single, constant yield $y$. In practice, bond pricing uses the discount rates from the spot curve at each cashflow's maturity. For cash flows $(t_j, CF_j)$ with $t_j > 0$:

$$
P^{\text{true}} = \sum_{j=1}^{N} CF_j \, e^{-z(t_j) \, t_j}
$$

Given a market price, the implied flat yield $y$ is the (unique) number that reproduces the same price under flat discounting:

$$
P^{\text{true}} = \sum_{j=1}^{N} CF_j \, e^{-y \, t_j}
$$

Define

$$
G(y) = \sum_{j=1}^{N} CF_j e^{-y t_j} - P^{\text{true}},
$$

then $G'(y) = -\sum_j t_j CF_j e^{-y t_j} < 0 \implies  G(y) $ is strictly decreasing, hence it has one real root. By setting $G(y)=0$, $y$ can be derived using numerical methods. Here is a simple example.

Price two 5-year annual coupon bonds, both with face 1000, and coupon rates of 0.10 and 0.02, respectively. Assume annual spot rates are as follows: $z_{1} = 0.03, z_{2} = 0.035, z_{3} = 0.04, z_{4} = 0.045, z_{5}=0.05$. Using continuous zero rates $r_{t} = \text{ln}(1+z_{t})$, prices are:

\begin{align*}
P_{A} &= \sum_{t=1}^{5}100e^{-r_{t}t}  + 1000e^{-r_{5}5}= 1225.07 \\
P_{B} &= \sum_{t=1}^{5}20e^{-r_{t}t} + 1000e^{-r_{5}5} = 871.84 \\
\end{align*}

with corresponding implied (annual) yields 4.8267\% and 4.9568\%.

I provide closed form expressions for a first-order and second-order approximation for flat $y$.

## First-Order

Expand $e^{-(y-z(t_j))t_j} \approx 1 - (y - z(t_j)) t_j$:

$$
0 \approx -\sum_{j=1}^{N} CF_j e^{-z(t_j) t_j} t_j (y - z(t_j)).
$$

and let

\begin{align*}
w_j &= \frac{CF_j e^{-z(t_j) t_j}}{P^{\text{true}}} \\
D &= \sum_{j=1}^{N} w_j t_j
\end{align*}

then the first-order approximation gives

$$
\boxed{
y \approx \bar{z} \equiv \frac{1}{D} \sum_{j=1}^{N} w_j \, t_j \, z(t_j)
}
$$

which can be interpreted as the PV-time-weighted average of the spot rates.

## Second-Order Approximation for the Equivalent Flat Yield $y$

Start from the exact pricing identity $G(y)=0$ under continuous compounding:

$$
\sum_{j=1}^{N} CF_j e^{-z_j t_j} = \sum_{j=1}^{N} CF_j e^{-y t_j},
\qquad z_j \equiv z(t_j)
$$

and let $ PV_j = CF_j e^{-z_j t_j} $, $ P = \sum_j PV_j $, then

$$
0 = \sum_{j=1}^{N} PV_j \left( e^{-(y - z_j)t_j} - 1 \right)
$$

I expand the exponential up to the second order

$$
e^{-(y - z_j)t_j} \approx 1 - (y - z_j)t_j + \tfrac{1}{2}(y - z_j)^2 t_j^2.
$$

Substituting into the previous expression gives

$$
0 \approx
-\sum_j PV_j\,t_j\,(y - z_j)
+ \tfrac{1}{2} \sum_j PV_j\,t_j^2\,(y - z_j)^2.
$$

and introducing new notation for present value based weights

\begin{align*}
w_j &= \frac{PV_j}{P} \\
M_k &\equiv \sum_j w_j\,t_j^{\,k},
\end{align*}


and mixed moments

\begin{align*}
B_1 &= \sum_j w_j t_j z_j, \\
B_2 &= \sum_j w_j t_j^2 z_j \\
C_2 &= \sum_j w_j t_j^2 z_j^2
\end{align*}

gives a clean quadratic equation in $y$:

$$
\frac{1}{2} M_2 y^2 - (M_1 + B_2) y + \left(B_1 + \tfrac{1}{2} C_2\right) \approx 0.
$$

$M_{1}$ is the PV-weighted average time (continuous duration). $M_2$ is PV-time-squared moment (related to convexity), and $B_1, B_2, C_2$ capture how the curve’s shape ($z_j$) interacts with the timing of cash flows.


Thus the closed form for the second order approximation is

$$
\boxed{
y^{(2)} \approx
\frac{(M_1 + B_2) -
\sqrt{(M_1 + B_2)^2 - 2M_2\!\left(B_1 + \tfrac{1}{2}C_2\right)}}
{M_2}.
}
$$

Note the following. First, if the curve is flat ($z_j = z_0$), this collapses to $y = z_0$. Second, when curvature terms are small, it reduces to the first-order approximation below.

$$
\boxed{
y^{(1)} = \frac{B_1}{M_1}
= \frac{\sum_j w_j t_j z_j}{\sum_j w_j t_j},
}
$$

The code demonstrates the numerical approach using scipy's fsolve.
I compare the approximations to the numerical YTMs below.

In [1]:
# minimal dependencies
import numpy as np
from scipy.optimize import fsolve


# price bond, assume continuous compounding
def price(f, c, z_t):
    """
    Compute price of a bond with face value f, coupon rate c, and spot rates z_t.

    Args:
      f : float
          Face value
      c : float
          Coupon rate (per year)
      z_t : array-like
          Spot rates per period, already in continuous compounding

    Returns:
      price : float
          Bond price under the provided spot curve
    """
    # number of periods
    T = len(z_t)

    # time index (1..T)
    t = np.arange(1, T + 1, dtype=float)

    # discount factors under continuous compounding
    df_t = np.exp(-z_t * t)

    # cashflows (coupon every period, principal at T)
    c_t = np.ones(T) * c * f
    c_t[-1] += f

    # discounted cashflow sum
    price = np.sum(df_t * c_t)

    # return price
    return price


# solve for YTM (continuous compounded flat yield)
def ytm(p, f, c, T, y_0=0.01):
    """
    Compute implied *continuous-compounded* YTM y from market price p,
    assuming a flat curve with rate y.

    Args:
      p : float
          Observed market price
      f : float
          Face value
      c : float
          Coupon rate
      T : int
          Number of coupon periods
      y_0 : float
          Initial guess for the solver (continuous rate)

    Returns:
      y : float
          Implied continuous-compounded YTM
    """
    # time index (1..T)
    t = np.arange(1, T + 1, dtype=float)

    # objective: price at flat y minus target price
    obj = lambda y: price(f, c, y * np.ones(T)) - p

    # solve for y (scalar)
    y = fsolve(obj, y_0)[0]

    # return y
    return y


# approximate implied y via PV-time moments (1st- and 2nd-order)
def y_approx(z_t, cf_t, p_true):
    """
    Compute 1st- and 2nd-order closed-form approximations of the
    continuous-compounded implied yield y using PV-time moments.

    Args:
      z_t   : array-like
              Spot rates at each cashflow time (continuous-compounded)
      cf_t  : array-like
              Cashflows at each time (same length as z_t)
      p_true: float
              True spot-curve price (sum cf_t * exp(-z_t * t))

    Returns:
      y_1, y_2 : tuple of floats
                 First-order y^{(1)} and second-order y^{(2)} (both continuous rates)
    """
    # validate inputs
    assert len(z_t) == len(cf_t)

    # number of periods
    T = len(z_t)

    # time index (1..T)
    t = np.arange(1, T + 1, dtype=float)

    # PV weights under the true curve
    w_t = cf_t * np.exp(-z_t * t) / p_true

    # moments and mixed moments
    D = np.sum(w_t * t)
    A2 = np.sum(w_t * t**2)
    B1 = np.sum(w_t * t * z_t)
    B2 = np.sum(w_t * t**2 * z_t)
    C2 = np.sum(w_t * t**2 * z_t**2)

    # first-order approximation: y^(1) = B1 / D
    y_1 = B1 / D

    # second-order approximation: solve (1/2)A2 y^2 - (D+B2) y + (B1 + 1/2 C2) = 0
    a = 0.5 * A2
    b = -(D + B2)
    c = (B1 + 0.5 * C2)

    # discriminant with numerical safety
    disc = b * b - 4 * a * c
    disc = max(disc, 0.0)

    # smaller (economically relevant) root
    y_2 = (-b - np.sqrt(disc)) / (2 * a) if a != 0 else y_1

    # return tuple
    return y_1, y_2


# build cashflows for face f, coupon c, horizon T
def cashflows(f, c, T):
    """
    Create a cashflow vector for a plain vanilla fixed rate bond.

    Args:
      f : float
          Face value
      c : float
          Coupon rate
      T : int
          Number of periods

    Returns:
      cf_t : np.ndarray
             Cashflows (length T)
    """
    # initialize with coupons
    cf_t = np.ones(T) * c * f

    # add principal at maturity
    cf_t[-1] += f

    # return vector
    return cf_t


# =========================
# Example: upward term structure (annual effective) -> continuous zeros
# =========================

# annual effective zeros
r_t = np.array([0.03, 0.035, 0.04, 0.045, 0.05], dtype=float)

# convert to continuous compounded zeros
z_t = np.log(1.0 + r_t)

# face value
f = 1000.0

# number of periods
T = len(z_t)

# cashflows for two bonds
cf_a = cashflows(f, 0.10, T)
cf_b = cashflows(f, 0.02, T)

# true spot-curve prices
p_a = price(f, 0.10, z_t)
p_b = price(f, 0.02, z_t)

# numerical implied YTMs (continuous)
y_num_a_cont = ytm(p_a, f, 0.10, T, y_0=0.04)
y_num_b_cont = ytm(p_b, f, 0.02, T, y_0=0.05)

# numerical implied YTMs (effective)
y_num_a_eff = np.exp(y_num_a_cont) - 1.0
y_num_b_eff = np.exp(y_num_b_cont) - 1.0

# closed-form approximations (continuous)
y1_a_cont, y2_a_cont = y_approx(z_t, cf_a, p_a)
y1_b_cont, y2_b_cont = y_approx(z_t, cf_b, p_b)

# closed-form approximations (effective)
y1_a_eff = np.exp(y1_a_cont) - 1.0
y2_a_eff = np.exp(y2_a_cont) - 1.0
y1_b_eff = np.exp(y1_b_cont) - 1.0
y2_b_eff = np.exp(y2_b_cont) - 1.0

# rounding for display
p_a_r, p_b_r = np.round(p_a, 4), np.round(p_b, 4)
y_num_a_eff_r = np.round(100 * y_num_a_eff, 4)
y_num_b_eff_r = np.round(100 * y_num_b_eff, 4)
y1_a_eff_r = np.round(100 * y1_a_eff, 4)
y2_a_eff_r = np.round(100 * y2_a_eff, 4)
y1_b_eff_r = np.round(100 * y1_b_eff, 4)
y2_b_eff_r = np.round(100 * y2_b_eff, 4)

# report
print(f"Bond A (10%): price = {p_a_r},  YTM (numerical) = {y_num_a_eff_r}%")
print(f"\tapprox YTM eff: 1st = {y1_a_eff_r}%,  2nd = {y2_a_eff_r}%\n")
print(f"Bond B (2%):  price = {p_b_r},  YTM (numerical) = {y_num_b_eff_r}%")
print(f"\tapprox YTM eff: 1st = {y1_b_eff_r}%,  2nd = {y2_b_eff_r}%")


Bond A (10%): price = 1225.073,  YTM (numerical) = 4.8263%
	approx YTM eff: 1st = 4.8242%,  2nd = 4.8263%

Bond B (2%):  price = 871.8355,  YTM (numerical) = 4.9567%
	approx YTM eff: 1st = 4.9562%,  2nd = 4.9567%


As expected, the second-order approximation shows marginal improvements over the first-order. Both are close to the numerical solution.

Given the precision of the numerical solver (exact to machine tolerance), why use the approximations? There are several reasons:

1. Speed: a numerical solver requires iterative evaluations of the pricing function, and computational cost grows with the number of bonds times iterations, whereas the approximation prices bonds in one pass ($\mathcal{O}(n)$ vs. $\mathcal{O}(n \times it)$).

2. Robustness: the closed form gives a stable solution, without any sensitivity to initial guess or risk of failure to converge. This is useful when a bond's cashflows are highly concentrated (e.g. very short or very long duration), such that small noise in the market price results in large changes in numerical yield.

3. Interpretability: the approximation exposes how the YTM estimate depends on the slope and curvature through the moment terms. Moreover, risk metrics like key rate duration or cashflow sensitivity ($\frac{\partial y}{\partial z(t)}$, $\frac{\partial y}{\partial CF_{j}}$) can be easily derived from the algebraic expressions but are much harder when $y$ is defined implicitly by a root.

Point 3 is especially clear with a parametric spot curve. Let

$$
z(t) = a + bt + ct^{2}
$$

for slope $b$ and curvature $c$. The first-order (linearized) implied flat yield is
$$
\boxed{
y^{(1)} \;=\; \frac{\sum_j w_j\,t_j\,z(t_j)}{\sum_j w_j\,t_j}
\;=\;
a \;+\; b\,\frac{M_2}{M_1} \;+\; c\,\frac{M_3}{M_1}.
}
$$

The second-order is more complicated. Let $z_j \equiv z(t_j)=a+b t_j+c t_j^2$. The second-order Taylor match gives a quadratic in $y$:

$$
\frac{1}{2}M_2\,y^2 \;-\; \big(M_1 + B_2\big)\,y \;+\; \Big(B_1 + \tfrac12 C_2\Big) \;\approx\; 0,
$$

which can be expressed entirely in $M_k$, $a, b, c$ as follows:


\begin{aligned}
B_1 &= \sum_j w_j\,t_j\,z_j \;=\; a\,M_1 + b\,M_2 + c\,M_3,\\[2pt]
B_2 &= \sum_j w_j\,t_j^2\,z_j \;=\; a\,M_2 + b\,M_3 + c\,M_4,\\[2pt]
C_2 &= \sum_j w_j\,t_j^2\,z_j^{\,2}
= a^2 M_2 + 2ab M_3 + (2ac + b^2) M_4 + 2bc M_5 + c^2 M_6.
\end{aligned}


The root is unwieldy, but working through the algebra gives

$$
\text{disc} = \big(M_1 + a M_2 + b M_3 + c M_4\big)^2 - M_2\Big(2(a M_1 + b M_2 + c M_3)+ a^2 M_2 + 2ab M_3 + (2ac + b^2) M_4 + 2bc M_5 + c^2 M_6\Big)
$$

$$
\boxed{y^{(2)} \approx \frac{\big(M_1 + a M_2 + b M_3 + c M_4\big)-\sqrt{\text{disc}}}{M_2}}
$$

The expressions check out. Note the following:

1. Flat Curve: $b=c=0$, thus $y^{(1)}=a$ and $y^{(2)}=a$ exactly.

2. Slope Loading: $b$ enters via $M_2/M_1$, so the effect is stronger for back-loaded cash flows (e.g. low coupon rate, zeros, generally face repayment dominates).

3. Curvature loading: $c$ enters via $M_3/M_1$ and, at second order, also via $M_4,M_5,M_6$.

There's a nice intuition. Back-loaded bonds that live more in the long end of the spot curve have higher implied YTM than a high coupon bond's YTM even if they are both priced fairly as the larger $M_2/M_1$ makes $y^{(1)}$ more sensitive to slope $b$. The code below demos the intuitions.

In [2]:
# generate a quadratic continuous spot curve z(t) = a + b t + c t^2
def z_param(a, b, c, T):
    """
    Build a vector of continuous-compounded spot rates z_t from
    the quadratic parametric curve z(t) = a + b t + c t^2.

    Args:
      a, b, c : floats
                Level, slope, curvature (per year, per year^2)
      T       : int
                Number of periods

    Returns:
      z_t : np.ndarray
            Continuous zeros at t=1..T
    """
    # time index (1..T)
    t = np.arange(1, T + 1, dtype=float)

    # quadratic spot curve at each t
    z_t = a + b * t + c * t**2

    # return curve
    return z_t


# compute Mk (PV-time) moments and y^(1) level/slope/curvature decomposition
def moments_and_decomp(z_t, cf_t, p_true, a, b, c):
    """
    Compute PV-time moments M1..M6 and decompose y^{(1)} = a + b*(M2/M1) + c*(M3/M1).

    Args:
      z_t   : array-like
              Continuous spot rates at each cashflow time
      cf_t  : array-like
              Cashflows aligned with z_t
      p_true: float
              True price under z_t
      a,b,c : floats
              Level, slope, curvature in z(t) = a + b t + c t^2

    Returns:
      report : dict
               Contains M1..M6, ratios M2/M1 and M3/M1, and contributions to y^{(1)}
    """
    # number of periods
    T = len(z_t)

    # time index (1..T)
    t = np.arange(1, T + 1, dtype=float)

    # PV weights under the true curve
    w_t = cf_t * np.exp(-z_t * t) / p_true

    # PV-time moments M1..M6
    M1 = np.sum(w_t * t)
    M2 = np.sum(w_t * t**2)
    M3 = np.sum(w_t * t**3)
    M4 = np.sum(w_t * t**4)
    M5 = np.sum(w_t * t**5)
    M6 = np.sum(w_t * t**6)

    # ratios for slope/curvature loadings
    r21 = M2 / M1
    r31 = M3 / M1

    # first-order y^(1) decomposition
    level_contrib = a
    slope_contrib = b * r21
    curv_contrib  = c * r31
    y1 = level_contrib + slope_contrib + curv_contrib

    # collect rounded results
    report = {"M1": np.round(M1, 6),
              "M2": np.round(M2, 6),
              "M3": np.round(M3, 6),
              "M4": np.round(M4, 6),
              "M5": np.round(M5, 6),
              "M6": np.round(M6, 6),
              "M2/M1": np.round(r21, 6),
              "M3/M1": np.round(r31, 6),
              "y1_level_pct": np.round(100 * level_contrib, 6),
              "y1_slope_pct": np.round(100 * slope_contrib, 6),
              "y1_curv_pct":  np.round(100 * curv_contrib, 6),
              "y1_total_pct": np.round(100 * y1, 6)
              }

    # return dictionary
    return report


# =========================
# Demo: intuition for front- vs back-loaded bonds
# =========================

# choose a quadratic continuous curve: level, slope, curvature
a = 0.03
b = 0.004
c = 0.0005

# number of periods
T = 5

# construct z_t from the parametric curve
z_t = z_param(a, b, c, T)

# face value
f = 1000.0

# define three cashflow profiles: zero (back-loaded),
# low-coupon (semi back-loaded), high-coupon (front-loaded)
cf_zero = cashflows(f, 0.00, T)
cf_low  = cashflows(f, 0.02, T)
cf_high = cashflows(f, 0.10, T)

# compute true prices under z_t
p_zero = price(f, 0.00, z_t)
p_low  = price(f, 0.02, z_t)
p_high = price(f, 0.10, z_t)

# compute first- and second-order y (continuous) for each profile
y1_zero, y2_zero = y_approx(z_t, cf_zero, p_zero)
y1_low,  y2_low  = y_approx(z_t, cf_low,  p_low)
y1_high, y2_high = y_approx(z_t, cf_high, p_high)

# convert to effective yields for display
y1_zero_eff = np.exp(y1_zero) - 1.0
y1_low_eff  = np.exp(y1_low)  - 1.0
y1_high_eff = np.exp(y1_high) - 1.0

# build moment-loadings reports
rep_zero = moments_and_decomp(z_t, cf_zero, p_zero, a, b, c)
rep_low  = moments_and_decomp(z_t, cf_low,  p_low,  a, b, c)
rep_high = moments_and_decomp(z_t, cf_high, p_high, a, b, c)

# print summary (prices and y^(1) effective)
print(f"Zero price = {np.round(p_zero, 4)},  y^(1)_eff = {np.round(100*y1_zero_eff, 4)}%")
print(f"Low 2% price = {np.round(p_low, 4)},   y^(1)_eff = {np.round(100*y1_low_eff, 4)}%")
print(f"High 10% price = {np.round(p_high, 4)},  y^(1)_eff = {np.round(100*y1_high_eff, 4)}%\n")

# print intuition: M2/M1 and M3/M1 drive slope and curvature loadings
print(f"Zero M2/M1 = {rep_zero['M2/M1']},  M3/M1 = {rep_zero['M3/M1']}")
print(f"Low 2% M2/M1 = {rep_low['M2/M1']},  M3/M1 = {rep_low['M3/M1']}")
print(f"High 10% M2/M1 = {rep_high['M2/M1']}, M3/M1 = {rep_high['M3/M1']}\n")

# print decomposition of y^{(1)} into level/slope/curvature contributions
print(f"Zero y^(1) % = level {rep_zero['y1_level_pct']}, slope {rep_zero['y1_slope_pct']}, curv {rep_zero['y1_curv_pct']}, total {rep_zero['y1_total_pct']}")
print(f"Low 2% y^(1) % = level {rep_low['y1_level_pct']},  slope {rep_low['y1_slope_pct']},  curv {rep_low['y1_curv_pct']},  total {rep_low['y1_total_pct']}")
print(f"High 10% y^(1) % = level {rep_high['y1_level_pct']}, slope {rep_high['y1_slope_pct']}, curv {rep_high['y1_curv_pct']}, total {rep_high['y1_total_pct']}\n")

# print differential sensitivities (derivatives of y^(1) wrt slope/curvature)
print(f"d y^(1)/d b = M2/M1  → Zero {rep_zero['M2/M1']},  Low {rep_low['M2/M1']},  High {rep_high['M2/M1']}")
print(f"d y^(1)/d c = M3/M1  → Zero {rep_zero['M3/M1']},  Low {rep_low['M3/M1']},  High {rep_high['M3/M1']}")

Zero price = 731.6156,  y^(1)_eff = 6.4494%
Low 2% price = 817.5426,   y^(1)_eff = 6.3744%
High 10% price = 1161.2505,  y^(1)_eff = 6.1499%

Zero M2/M1 = 5.0,  M3/M1 = 25.0
Low 2% M2/M1 = 4.908585,  M3/M1 = 24.320138
High 10% M2/M1 = 4.634954, M3/M1 = 22.285106

Zero y^(1) % = level 3.0, slope 2.0, curv 1.25, total 6.25
Low 2% y^(1) % = level 3.0,  slope 1.963434,  curv 1.216007,  total 6.179441
High 10% y^(1) % = level 3.0, slope 1.853981, curv 1.114255, total 5.968237

d y^(1)/d b = M2/M1  → Zero 5.0,  Low 4.908585,  High 4.634954
d y^(1)/d c = M3/M1  → Zero 25.0,  Low 24.320138,  High 22.285106


A zero shows larger $M_2/M_1$ and $M_3/M_1$ than a front loaded bond. The decomposition of $y^{(1)}$ into level, slope, and curvature shows back-loaded bonds' exposure to slope/curvature.